In [1]:
# 1) Setup & Imports
import os
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
 )
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
PREPROCESSED_DIR = Path(r"C:\Users\buck\Napplee\StressClassification\data\preprocessed")
DATA_ROOT = PREPROCESSED_DIR.parent
CONCATENATED_DIR = DATA_ROOT / "concatenated"
FEATURE_DIR = Path(r"C:\Users\buck\Napplee\StressClassification\data\features")
FEATURE_FILE = FEATURE_DIR / "features_hrv.csv"
ARTIFACTS_DIR = Path("artifacts")
REPORTS_DIR = Path("reports")

for p in [FEATURE_DIR, ARTIFACTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Preprocessed directory: {PREPROCESSED_DIR}")
print(f"Concatenated directory: {CONCATENATED_DIR}")
print(f"Feature file output: {FEATURE_FILE}")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")

Preprocessed directory: C:\Users\buck\Napplee\StressClassification\data\preprocessed
Concatenated directory: C:\Users\buck\Napplee\StressClassification\data\concatenated
Feature file output: C:\Users\buck\Napplee\StressClassification\data\features\features_hrv.csv
Artifacts directory: C:\Users\buck\Napplee\StressClassification\artifacts


In [2]:
def discover_data_files(data_dir: Path):
    patterns = ["*.csv", "*.xlsx", "*.xls", "*.json"]
    files = []
    for pattern in patterns:
        files.extend(data_dir.rglob(pattern))
    return sorted([p for p in files if p.is_file()])


def load_single_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported file format: {path}")


candidate_dirs = []
if CONCATENATED_DIR.exists():
    candidate_dirs.append(CONCATENATED_DIR)
if PREPROCESSED_DIR.exists():
    candidate_dirs.append(PREPROCESSED_DIR)

if not candidate_dirs:
    raise FileNotFoundError(
        "Khong tim thay thu muc du lieu nao hop le (concatenated/preprocessed)."
    )

all_candidates = []
for base_dir in candidate_dirs:
    files_here = discover_data_files(base_dir)
    all_candidates.extend([(base_dir, fp) for fp in files_here])

npy_files = sorted(PREPROCESSED_DIR.rglob("*.npy")) if PREPROCESSED_DIR.exists() else []

# Loai bo file metadata thong ke de tranh doc nham vao raw dataframe
metadata_names = {"preprocessing_stats.json"}
raw_files = [(base_dir, fp) for base_dir, fp in all_candidates if fp.name.lower() not in metadata_names]
skipped_metadata = [(base_dir, fp) for base_dir, fp in all_candidates if fp.name.lower() in metadata_names]

print(f"Tong so file tabular tim thay: {len(raw_files)}")
print(f"Tong so file npy tim thay trong preprocessed: {len(npy_files)}")
if skipped_metadata:
    print("Bo qua file metadata:", [str(p.name) for _, p in skipped_metadata])

frames = []
failed_files = []
skipped_non_waveform = []

for base_dir, fpath in raw_files:
    try:
        df_i = load_single_file(fpath)
        if df_i is None or df_i.empty:
            continue

        df_i = df_i.copy()
        if "source_file" not in df_i.columns:
            rel = str(fpath.relative_to(base_dir))
            df_i["source_file"] = f"{base_dir.name}/{rel}"

        # Chi giu file co cot lien quan den waveform/feature train
        accepted_markers = {"Time", "Voltage", "Peak", "label", "Label", "stress", "target", "class"}
        if len(set(df_i.columns).intersection(accepted_markers)) == 0:
            skipped_non_waveform.append(str(fpath))
            continue

        frames.append(df_i)
    except Exception as ex:
        failed_files.append((str(fpath), str(ex)))

raw_df = pd.concat(frames, axis=0, ignore_index=True) if frames else pd.DataFrame()
print("\n=== Raw Data Summary ===")
print(f"Shape: {raw_df.shape}")
if not raw_df.empty:
    print(f"Columns: {raw_df.columns.tolist()}")
    print("\nDtypes:")
    print(raw_df.dtypes)
    print("\nTop missing values:")
    print(raw_df.isna().sum().sort_values(ascending=False).head(20))
else:
    print("Khong co bang du lieu tabular hop le sau khi loc metadata.")

if skipped_non_waveform:
    print("\nBo qua file khong phai waveform/feature table:")
    for fp in skipped_non_waveform[:10]:
        print(f"- {fp}")

if failed_files:
    print("\nCanh bao: mot so file khong doc duoc:")
    for fp, err in failed_files[:10]:
        print(f"- {fp}: {err}")

raw_df.head() if not raw_df.empty else pd.DataFrame()

Tong so file tabular tim thay: 686
Tong so file npy tim thay trong preprocessed: 6
Bo qua file metadata: ['preprocessing_stats.json']

=== Raw Data Summary ===
Shape: (267927516, 4)
Columns: ['Time', 'Voltage', 'Peak', 'source_file']

Dtypes:
Time           float64
Voltage        float64
Peak             int64
source_file     object
dtype: object

Top missing values:
Time           0
Voltage        0
Peak           0
source_file    0
dtype: int64


,Time,Voltage,Peak,source_file
0,0.000000,0.947944,3,concatenated/mixed_0\mixed_0_001.csv
1,0.003906,0.970939,0,concatenated/mixed_0\mixed_0_001.csv
2,0.007812,0.914415,0,concatenated/mixed_0\mixed_0_001.csv
3,0.011719,0.601671,0,concatenated/mixed_0\mixed_0_001.csv
4,0.015625,0.548172,0,concatenated/mixed_0\mixed_0_001.csv


In [3]:
# Raw HR sanity check from waveform peaks (sampled sources)
if 'raw_df' not in globals() or raw_df.empty:
    raise ValueError("raw_df is missing. Run data loading cell first.")

def _infer_label_from_source(path_str: str) -> int:
    src = str(path_str).lower()
    patterns = [r"(?:pure|mixed)[_/](\d)", r"(?:pure|mixed)_(\d)", r"[/_](\d)(?:[/_]|$)"]
    for p in patterns:
        m = re.search(p, src)
        if m:
            return int(m.group(1))
    return -1

# Sample a limited number of sources per label to keep runtime reasonable
MAX_SOURCES_PER_LABEL = 12
labels_to_check = [0, 1, 2, 3]

src_df = raw_df[['source_file']].drop_duplicates().copy()
src_df['label'] = src_df['source_file'].apply(_infer_label_from_source)
src_df = src_df[src_df['label'].isin(labels_to_check)]

sampled_sources = []
for lbl in labels_to_check:
    src_lbl = src_df[src_df['label'] == lbl]['source_file'].tolist()
    if not src_lbl:
        continue
    if len(src_lbl) > MAX_SOURCES_PER_LABEL:
        src_lbl = list(pd.Series(src_lbl).sample(n=MAX_SOURCES_PER_LABEL, random_state=RANDOM_STATE))
    sampled_sources.extend(src_lbl)

df_sample = raw_df[raw_df['source_file'].isin(sampled_sources)].copy()
df_sample['Time'] = pd.to_numeric(df_sample['Time'], errors='coerce')
df_sample['Peak'] = pd.to_numeric(df_sample['Peak'], errors='coerce')
df_sample = df_sample.dropna(subset=['Time', 'Peak']).copy()

hr_rows = []
for src, g in df_sample.groupby('source_file', sort=False):
    g = g.sort_values('Time')
    peak_times = g.loc[g['Peak'] == 3, 'Time'].values
    if len(peak_times) < 4:
        continue
    rr_ms = np.diff(peak_times) * 1000.0
    rr_ms = rr_ms[(rr_ms >= 300.0) & (rr_ms <= 2000.0)]
    if len(rr_ms) < 3:
        continue
    mean_rr = float(np.mean(rr_ms))
    hr_bpm = 60000.0 / mean_rr if mean_rr > 0 else np.nan
    hr_rows.append({
        'source_file': src,
        'label': _infer_label_from_source(src),
        'rr_count': int(len(rr_ms)),
        'mean_rr_ms': round(mean_rr, 3),
        'mean_hr_bpm': round(hr_bpm, 3),
    })

hr_df = pd.DataFrame(hr_rows)
if hr_df.empty:
    print("Raw HR check: no valid peak sequences found in sampled sources.")
else:
    print("Raw HR check (sampled sources):")
    display(hr_df.sort_values(['label', 'mean_hr_bpm']))
    print("\nSummary by label:")
    display(hr_df.groupby('label')['mean_hr_bpm'].agg(['count', 'mean', 'min', 'max']).round(3))
    print("\nCount > 110 bpm by label:")
    display(hr_df[hr_df['mean_hr_bpm'] > 110].groupby('label')['mean_hr_bpm'].count())

Raw HR check (sampled sources):


,source_file,label,rr_count,mean_rr_ms,mean_hr_bpm
11,concatenated/pure_0\concat_0_058.csv,0,2480,725.706,82.678
7,concatenated/pure_0\concat_0_009.csv,0,2537,709.215,84.601
17,concatenated/pure_0\concat_0_130.csv,0,1830,699.703,85.751
14,concatenated/pure_0\concat_0_082.csv,0,1844,694.810,86.354
8,concatenated/pure_0\concat_0_010.csv,0,2632,683.675,87.761
9,concatenated/pure_0\concat_0_028.csv,0,2657,677.298,88.587
13,concatenated/pure_0\concat_0_081.csv,0,2669,674.142,89.002
16,concatenated/pure_0\concat_0_105.csv,0,2673,673.228,89.123
12,concatenated/pure_0\concat_0_076.csv,0,1903,673.178,89.129
15,concatenated/pure_0\concat_0_092.csv,0,2683,670.668,89.463



Summary by label:


,count,mean,min,max
label,,,,
0,12,87.896,82.678,91.415
1,12,88.634,82.388,92.586
2,12,90.412,86.202,93.576
3,12,95.523,89.709,100.104



Count > 110 bpm by label:


Series([], Name: mean_hr_bpm, dtype: int64)

In [4]:
REQUIRED_COLS = ["Time", "Voltage", "Peak", "source_file"]
LABEL_CANDIDATES = ["label", "Label", "stress", "stress_level", "target", "class"]

# Data quality first: longer window helps stabilize LF/HF
WINDOW_SEC = 120.0
MIN_RR_COUNT = 5
MIN_WINDOW_DURATION_SEC = 20.0
MIN_PEAK_RATE = 0.4
MAX_PEAK_RATE = 8.0
MIN_HR_BPM = 40.0
MAX_HR_BPM = 190.0
MIN_RR_VALID_RATIO = 0.25
MAX_SOURCE_SHARE_PER_LABEL = 0.30
MAX_WINDOWS_PER_SOURCE_PER_LABEL = 20
OVERSAMPLE_TRIGGER_RATIO = 2.5
LABEL_FROM_PATH_ONLY = True
MIN_WINDOWS_PER_SOURCE = 4
CLASSWISE_Q_LOW = 0.02
CLASSWISE_Q_HIGH = 0.98
TARGET_LABEL_IMBALANCE_MAX = 1.40
MIN_ROWS_PER_LABEL_AFTER_OUTLIER = 280
MAX_SOURCES_PER_LABEL_FOR_FEATURES = 60

def find_label_column(df_in: pd.DataFrame):
    for c in LABEL_CANDIDATES:
        if c in df_in.columns:
            return c
    return None

def load_feature_candidates() -> pd.DataFrame:
    data_root = PREPROCESSED_DIR.parent
    candidates = [
        FEATURE_FILE,
        data_root / "hrv_features_label.csv",
        data_root / "hrv_features.csv",
    ]

    for fp in candidates:
        if fp.exists() and fp.is_file():
            try:
                tmp = pd.read_csv(fp)
                if not tmp.empty:
                    print(f"Fallback: su dung feature file co san -> {fp}")
                    return tmp.copy()
            except Exception as ex:
                print(f"Khong doc duoc {fp}: {ex}")

    return pd.DataFrame()

def standardize_features_df(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    out = out.loc[:, ~out.columns.astype(str).str.startswith("Unnamed:")].copy()

    label_col = find_label_column(out)
    if label_col is None:
        raise ValueError(
            "Khong tim thay cot label trong feature dataset. Can mot trong cac cot: "
            + ", ".join(LABEL_CANDIDATES)
        )

    if label_col != "label":
        out = out.rename(columns={label_col: "label"})

    out = out.dropna(subset=["label"]).copy()
    if out.empty:
        raise ValueError("Feature dataset rong sau khi loai bo dong thieu label.")

    if "source_file" not in out.columns:
        out["source_file"] = "precomputed_features"
    if "window_id" not in out.columns:
        out["window_id"] = np.arange(len(out))

    if out["label"].dtype == "O" or str(out["label"].dtype).startswith("category"):
        out["label"] = out["label"].astype(str).str.strip()
    else:
        out["label"] = pd.to_numeric(out["label"], errors="coerce")

    out = out.dropna(subset=["label"]).copy()
    if out.empty:
        raise ValueError("Khong con mau nao sau khi chuan hoa cot label.")

    return out

def load_from_preprocessed_npy(preprocessed_dir: Path) -> pd.DataFrame:
    split_pairs = [("train", "X_train.npy", "y_train.npy"), ("val", "X_val.npy", "y_val.npy"), ("test", "X_test.npy", "y_test.npy")]
    parts = []

    for split_name, x_name, y_name in split_pairs:
        x_path = preprocessed_dir / x_name
        y_path = preprocessed_dir / y_name
        if not (x_path.exists() and y_path.exists()):
            continue

        X_split = np.load(x_path, allow_pickle=True)
        y_split = np.load(y_path, allow_pickle=True)

        if len(X_split.shape) > 2:
            X_split = X_split.reshape(X_split.shape[0], -1)
        if len(y_split.shape) > 1:
            y_split = y_split.reshape(-1)

        if len(X_split) != len(y_split):
            raise ValueError(f"Kich thuoc X/y khong khop o split {split_name}: {len(X_split)} vs {len(y_split)}")

        col_names = [f"feat_{i:04d}" for i in range(X_split.shape[1])]
        df_split = pd.DataFrame(X_split, columns=col_names)
        df_split["label"] = y_split
        df_split["source_file"] = f"preprocessed_{split_name}"
        df_split["window_id"] = np.arange(len(df_split))
        parts.append(df_split)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, axis=0, ignore_index=True)

def clean_waveform_group(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("Time").copy()
    g = g.drop_duplicates(subset=["Time"], keep="first")

    g = g[g["Time"].notna()].copy()
    g = g[g["Time"] >= 0].copy()
    g = g[g["Time"].diff().fillna(1) > 0].copy()

    g["Peak"] = pd.to_numeric(g["Peak"], errors="coerce")

    v = pd.to_numeric(g["Voltage"], errors="coerce")
    q_low, q_high = np.nanquantile(v, [0.005, 0.995])
    g["Voltage"] = v.clip(lower=q_low, upper=q_high)

    return g.dropna(subset=["Time", "Voltage", "Peak"]).copy()

def deduplicate_features(df_feat: pd.DataFrame) -> pd.DataFrame:
    key_cols = ["label", "sdnn_ms", "rmssd_ms", "pnn50", "lf_hf_ratio", "heart_rate_bpm"]
    missing = [c for c in key_cols if c not in df_feat.columns]
    if missing:
        return df_feat

    tmp = df_feat.copy()
    for c in ["sdnn_ms", "rmssd_ms", "pnn50", "lf_hf_ratio", "heart_rate_bpm"]:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce").round(3)

    before = len(tmp)
    tmp = tmp.drop_duplicates(subset=key_cols, keep="first").copy()
    removed = before - len(tmp)
    print(f"Deduplicate features: removed {removed}/{before} rows ({removed / max(before, 1):.2%})")
    return tmp

def cap_source_dominance(df_feat: pd.DataFrame, max_share_per_label: float = MAX_SOURCE_SHARE_PER_LABEL) -> pd.DataFrame:
    if "source_file" not in df_feat.columns:
        return df_feat

    pieces = []
    dropped_total = 0

    for label_val, g_label in df_feat.groupby("label", sort=False):
        total = len(g_label)
        share_cap = max(int(np.floor(total * max_share_per_label)), 8)
        cap = min(share_cap, MAX_WINDOWS_PER_SOURCE_PER_LABEL)
        for src, g_src in g_label.groupby("source_file", sort=False):
            if len(g_src) > cap:
                g_src = g_src.sample(n=cap, random_state=RANDOM_STATE)
                dropped_total += max(len(g_label[g_label["source_file"] == src]) - cap, 0)
            pieces.append(g_src)

    out = pd.concat(pieces, axis=0, ignore_index=True) if pieces else df_feat.copy()
    out = out.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Cap source dominance: removed {dropped_total} rows")
    return out

def prune_sparse_sources(df_feat: pd.DataFrame, min_windows_per_source: int = MIN_WINDOWS_PER_SOURCE) -> pd.DataFrame:
    if "source_file" not in df_feat.columns:
        return df_feat

    counts = df_feat["source_file"].value_counts()
    keep_sources = counts[counts >= min_windows_per_source].index
    out = df_feat[df_feat["source_file"].isin(keep_sources)].copy()
    removed = len(df_feat) - len(out)
    print(f"Prune sparse sources: removed {removed} rows (min_windows_per_source={min_windows_per_source})")
    return out if not out.empty else df_feat

def filter_classwise_hrv_outliers(df_feat: pd.DataFrame, q_low: float = CLASSWISE_Q_LOW, q_high: float = CLASSWISE_Q_HIGH) -> pd.DataFrame:
    core_cols = ["sdnn_ms", "rmssd_ms", "pnn50", "lf_hf_ratio"]
    if any(c not in df_feat.columns for c in core_cols):
        return df_feat

    pieces = []
    removed_total = 0
    for label_val, g in df_feat.groupby("label", sort=False):
        def build_mask(g_in: pd.DataFrame, lo_q: float, hi_q: float) -> pd.Series:
            m = pd.Series(True, index=g_in.index)
            for c in core_cols:
                s = pd.to_numeric(g_in[c], errors="coerce")
                if s.notna().sum() < 20:
                    continue
                lo = float(s.quantile(lo_q))
                hi = float(s.quantile(hi_q))
                m &= s.between(lo, hi, inclusive="both") | s.isna()
            return m

        mask = build_mask(g, q_low, q_high)
        kept = g.loc[mask].copy()

        if len(kept) < MIN_ROWS_PER_LABEL_AFTER_OUTLIER and len(g) >= MIN_ROWS_PER_LABEL_AFTER_OUTLIER:
            # Loosen outlier filter for classes losing too many rows
            mask = build_mask(g, 0.01, 0.99)
            kept = g.loc[mask].copy()

        if len(kept) < MIN_ROWS_PER_LABEL_AFTER_OUTLIER and len(g) >= MIN_ROWS_PER_LABEL_AFTER_OUTLIER:
            mask = build_mask(g, 0.005, 0.995)
            kept = g.loc[mask].copy()

        removed_total += len(g) - len(kept)
        pieces.append(kept)

    out = pd.concat(pieces, axis=0, ignore_index=True) if pieces else df_feat.copy()
    print(f"Classwise HRV outlier filter: removed {removed_total} rows (q=[{q_low:.2f}, {q_high:.2f}])")
    return out if not out.empty else df_feat

def rebalance_labels_datafirst(df_feat: pd.DataFrame, target_ratio: float = TARGET_LABEL_IMBALANCE_MAX) -> pd.DataFrame:
    if "label" not in df_feat.columns or df_feat.empty:
        return df_feat

    counts = df_feat["label"].value_counts()
    min_count = int(counts.min())
    max_allowed = int(np.floor(min_count * target_ratio))

    parts = []
    removed = 0
    for label_val, g in df_feat.groupby("label", sort=False):
        if len(g) > max_allowed:
            g_keep = g.sample(n=max_allowed, random_state=RANDOM_STATE)
            removed += len(g) - len(g_keep)
            parts.append(g_keep)
        else:
            parts.append(g)

    out = pd.concat(parts, axis=0, ignore_index=True)
    out = out.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    new_counts = out["label"].value_counts()
    new_ratio = float(new_counts.max() / max(new_counts.min(), 1))
    print(f"Label rebalance (data-first): removed {removed} rows, new imbalance={new_ratio:.3f}")
    return out

def apply_feature_quality_filters(df_feat: pd.DataFrame) -> pd.DataFrame:
    df_q = df_feat.copy()
    before = len(df_q)

    # Stage 1: hard physiological constraints
    rules = pd.Series(True, index=df_q.index)
    rules &= df_q["duration_sec"].between(MIN_WINDOW_DURATION_SEC, 180.0, inclusive="both")
    rules &= df_q["heart_rate_bpm"].between(MIN_HR_BPM, MAX_HR_BPM, inclusive="both")
    rules &= df_q["sdnn_ms"].fillna(0) <= 300.0
    rules &= df_q["rmssd_ms"].fillna(0) <= 400.0
    df_q = df_q.loc[rules].copy()

    if df_q.empty:
        print("Canh bao: hard filter loai het du lieu. Se giu lai data truoc loc de tranh crash.")
        return df_feat.copy()

    # Stage 2: rr_valid_ratio adaptive threshold
    if "rr_valid_ratio" in df_q.columns:
        rr_thr = MIN_RR_VALID_RATIO
        rr_mask = df_q["rr_valid_ratio"].fillna(0) >= rr_thr
        rr_keep = int(rr_mask.sum())
        rr_min_keep = max(int(0.2 * len(df_q)), 200)
        if rr_keep < rr_min_keep:
            rr_thr = float(df_q["rr_valid_ratio"].quantile(0.20))
            rr_thr = min(max(rr_thr, 0.05), MIN_RR_VALID_RATIO)
            rr_mask = df_q["rr_valid_ratio"].fillna(0) >= rr_thr
            print(f"Noi long rr_valid_ratio threshold -> {rr_thr:.3f}")
        df_q = df_q.loc[rr_mask].copy()

    if df_q.empty:
        print("Canh bao: rr_valid_ratio filter loai het du lieu. Se giu lai data sau hard filter.")
        df_q = df_feat.loc[rules].copy()

    # Stage 3: peak-rate filter with robust fallback
    if "peak_rate_per_sec" in df_q.columns:
        pr_mask = df_q["peak_rate_per_sec"].between(MIN_PEAK_RATE, MAX_PEAK_RATE, inclusive="both")
        pr_keep = int(pr_mask.sum())
        pr_min_keep = max(int(0.3 * len(df_q)), 200)
        if pr_keep < pr_min_keep:
            lo = float(df_q["peak_rate_per_sec"].quantile(0.01))
            hi = float(df_q["peak_rate_per_sec"].quantile(0.99))
            pr_mask = df_q["peak_rate_per_sec"].between(lo, hi, inclusive="both")
            print(f"Noi long peak_rate range -> [{lo:.3f}, {hi:.3f}]")
        if pr_mask.any():
            df_q = df_q.loc[pr_mask].copy()

    removed = before - len(df_q)
    print(f"Quality filter: removed {removed}/{before} windows ({removed / max(before, 1):.2%})")

    if df_q.empty:
        print("Canh bao: quality filter loai het du lieu. Se giu lai data truoc loc de tranh crash.")
        return df_feat.copy()

    num_cols = [c for c in df_q.select_dtypes(include=[np.number]).columns if c not in {"label", "window_id"}]
    for c in num_cols:
        lo, hi = df_q[c].quantile([0.005, 0.995])
        df_q[c] = df_q[c].clip(lo, hi)

    return df_q

# Mode 1: raw waveform in preprocessed or concatenated
if set(REQUIRED_COLS).issubset(set(raw_df.columns)):
    print("Da tim thay du lieu raw waveform -> trich xuat feature HRV (data-first mode).")
    df = raw_df
    for c in ["Time", "Voltage", "Peak"]:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # Avoid dropna on the full dataframe to reduce memory pressure.
    # Per-group cleaning will remove NaNs later.
    if df.empty:
        raise ValueError("Du lieu raw rong sau khi lam sach cot Time/Voltage/Peak/source_file.")

    def _infer_label_from_source(path_str: str) -> int:
        src = str(path_str).lower()
        patterns = [r"(?:pure|mixed)[_/](\d)", r"(?:pure|mixed)_(\d)", r"[/_](\d)(?:[/_]|$)"]
        for p in patterns:
            m = re.search(p, src)
            if m:
                return int(m.group(1))
        return -1

    def infer_label(group_df: pd.DataFrame, source_name: str):
        if not LABEL_FROM_PATH_ONLY:
            for col in LABEL_CANDIDATES:
                if col in group_df.columns and group_df[col].notna().any():
                    v = group_df[col].dropna().mode().iloc[0]
                    try:
                        return int(float(v))
                    except Exception:
                        return str(v)

        label_val = _infer_label_from_source(source_name)
        return label_val if label_val >= 0 else np.nan

    def safe_skew(x: np.ndarray) -> float:
        return float(pd.Series(x).skew()) if len(x) >= 3 else np.nan

    def safe_kurt(x: np.ndarray) -> float:
        return float(pd.Series(x).kurt()) if len(x) >= 4 else np.nan

    def rr_frequency_features(rr_ms: np.ndarray):
        if len(rr_ms) < 4:
            return np.nan, np.nan, np.nan, np.nan
        rr_s = rr_ms / 1000.0
        t = np.cumsum(rr_s) - rr_s[0]
        if t[-1] <= 0:
            return np.nan, np.nan, np.nan, np.nan
        fs = 4.0
        t_uniform = np.arange(0, t[-1], 1 / fs)
        if len(t_uniform) < 8:
            return np.nan, np.nan, np.nan, np.nan

        rr_interp = np.interp(t_uniform, t, rr_ms)
        rr_detrended = rr_interp - np.nanmean(rr_interp)
        fft_vals = np.fft.rfft(rr_detrended)
        freqs = np.fft.rfftfreq(len(rr_detrended), d=1 / fs)
        psd = (np.abs(fft_vals) ** 2) / max(len(rr_detrended), 1)

        lf_mask = (freqs >= 0.04) & (freqs < 0.15)
        hf_mask = (freqs >= 0.15) & (freqs <= 0.40)
        total_mask = (freqs >= 0.04) & (freqs <= 0.40)

        lf_power = float(np.trapezoid(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else np.nan
        hf_power = float(np.trapezoid(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else np.nan
        total_power = float(np.trapezoid(psd[total_mask], freqs[total_mask])) if np.any(total_mask) else np.nan
        lf_hf_ratio = (lf_power / hf_power) if (pd.notna(lf_power) and pd.notna(hf_power) and hf_power > 0) else np.nan
        return lf_power, hf_power, lf_hf_ratio, total_power

    def extract_features_from_group(g: pd.DataFrame, source_name: str, window_id=np.nan):
        g = clean_waveform_group(g)
        duration_sec = float(g["Time"].max() - g["Time"].min()) if len(g) else 0.0
        if len(g) < 20 or duration_sec < MIN_WINDOW_DURATION_SEC:
            return None, "low_window_duration_or_length"

        peak_times = g.loc[g["Peak"] == 3, "Time"].values
        if len(peak_times) < 4:
            return None, "too_few_peaks"

        rr_raw = np.diff(peak_times) * 1000.0
        rr_ms = rr_raw[(rr_raw >= 300.0) & (rr_raw <= 2000.0)]
        rr_valid_ratio = float(len(rr_ms) / max(len(rr_raw), 1))
        if len(rr_ms) < MIN_RR_COUNT:
            return None, "too_few_rr_after_filter"

        rmssd = np.sqrt(np.mean(np.diff(rr_ms) ** 2)) if len(rr_ms) >= 2 else np.nan
        pnn50 = float(np.mean(np.abs(np.diff(rr_ms)) > 50.0) * 100.0) if len(rr_ms) >= 2 else np.nan
        mean_rr_ms = float(np.mean(rr_ms))
        sdnn_ms = float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else np.nan
        hr_bpm = 60000.0 / mean_rr_ms if mean_rr_ms > 0 else np.nan
        n_peaks = int(np.sum(g["Peak"] == 3))
        peak_rate = n_peaks / duration_sec if duration_sec > 0 else np.nan
        lf_power, hf_power, lf_hf_ratio, total_power = rr_frequency_features(rr_ms)

        row = {
            "source_file": source_name,
            "window_id": window_id,
            "label": infer_label(g, source_name),
            "mean_rr_ms": mean_rr_ms,
            "median_rr_ms": float(np.median(rr_ms)),
            "sdnn_ms": sdnn_ms,
            "std_rr_ms": sdnn_ms,
            "rmssd_ms": float(rmssd),
            "pnn50": float(pnn50),
            "heart_rate_bpm": float(hr_bpm),
            "n_peaks": n_peaks,
            "duration_sec": duration_sec,
            "peak_rate_per_sec": float(peak_rate),
            "rr_valid_ratio": rr_valid_ratio,
            "voltage_mean": float(g["Voltage"].mean()),
            "voltage_std": float(g["Voltage"].std(ddof=1)) if len(g) > 1 else np.nan,
            "voltage_min": float(g["Voltage"].min()),
            "voltage_max": float(g["Voltage"].max()),
            "voltage_skew": safe_skew(g["Voltage"].values),
            "voltage_kurtosis": safe_kurt(g["Voltage"].values),
            "lf_power": lf_power,
            "hf_power": hf_power,
            "lf_hf_ratio": lf_hf_ratio,
            "total_power": total_power,
        }
        return row, None

    sampled_sources_set = None
    source_name_list = []
    if "raw_files" in globals() and raw_files:
        for base_dir, fpath in raw_files:
            rel = str(fpath.relative_to(base_dir)).replace('\\', '/')
            source_name_list.append(f"{base_dir.name}/{rel}")
    else:
        # Fallback only when raw_files is unavailable
        source_name_list = pd.Series(df["source_file"].astype(str)).drop_duplicates().tolist()

    if MAX_SOURCES_PER_LABEL_FOR_FEATURES is not None and MAX_SOURCES_PER_LABEL_FOR_FEATURES > 0:
        src_df = pd.DataFrame({"source_file": source_name_list})
        src_df["label"] = src_df["source_file"].apply(_infer_label_from_source)
        src_df = src_df[src_df["label"].isin([0, 1, 2, 3])].copy()
        sampled_sources = []
        for lbl in sorted(src_df["label"].unique().tolist()):
            src_list = src_df[src_df["label"] == lbl]["source_file"].tolist()
            if len(src_list) > MAX_SOURCES_PER_LABEL_FOR_FEATURES:
                src_list = list(pd.Series(src_list).sample(n=MAX_SOURCES_PER_LABEL_FOR_FEATURES, random_state=RANDOM_STATE))
            sampled_sources.extend(src_list)
        if sampled_sources:
            sampled_sources_set = set(sampled_sources)
            print(f"Using sampled sources for feature extraction: {len(sampled_sources_set)} files")
    feature_rows = []
    skip_log = {
        "low_window_duration_or_length": 0,
        "too_few_peaks": 0,
        "too_few_rr_after_filter": 0,
    }

    if "raw_files" in globals() and raw_files:
        process_files = []
        for base_dir, fpath in raw_files:
            rel = str(fpath.relative_to(base_dir)).replace('\\', '/')
            source_name = f"{base_dir.name}/{rel}"
            if sampled_sources_set is not None and source_name not in sampled_sources_set:
                continue
            process_files.append((source_name, fpath))

        for source_name, fpath in process_files:
            try:
                g = load_single_file(Path(fpath))
            except Exception:
                skip_log["read_error"] = skip_log.get("read_error", 0) + 1
                continue
            if g is None or g.empty:
                skip_log["empty_file"] = skip_log.get("empty_file", 0) + 1
                continue
            g = g.copy()
            g["source_file"] = source_name
            duration = float(g["Time"].max() - g["Time"].min())
            if duration > WINDOW_SEC * 1.5:
                g2 = g.copy()
                g2["_window"] = ((g2["Time"] - g2["Time"].min()) // WINDOW_SEC).astype(int)
                for wv, gw in g2.groupby("_window", sort=False):
                    row, reason = extract_features_from_group(gw, source_name, window_id=int(wv))
                    if row is not None:
                        feature_rows.append(row)
                    else:
                        skip_log[reason] = skip_log.get(reason, 0) + 1
                continue

            row, reason = extract_features_from_group(g, source_name)
            if row is not None:
                feature_rows.append(row)
            else:
                skip_log[reason] = skip_log.get(reason, 0) + 1
    else:
        for source_name, g in df.groupby("source_file", sort=False):
            source_name = str(source_name)
            if sampled_sources_set is not None and source_name not in sampled_sources_set:
                continue
            duration = float(g["Time"].max() - g["Time"].min())
            if duration > WINDOW_SEC * 1.5:
                g2 = g.copy()
                g2["_window"] = ((g2["Time"] - g2["Time"].min()) // WINDOW_SEC).astype(int)
                for wv, gw in g2.groupby("_window", sort=False):
                    row, reason = extract_features_from_group(gw, source_name, window_id=int(wv))
                    if row is not None:
                        feature_rows.append(row)
                    else:
                        skip_log[reason] = skip_log.get(reason, 0) + 1
                continue

            row, reason = extract_features_from_group(g, source_name)
            if row is not None:
                feature_rows.append(row)
            else:
                skip_log[reason] = skip_log.get(reason, 0) + 1

    features_df = pd.DataFrame(feature_rows)
    if features_df.empty:
        raise ValueError("Khong trich xuat duoc feature tu raw waveform.")

    print("Skip log:", skip_log)
    features_df = features_df.dropna(subset=["label"]).copy()
    features_df = apply_feature_quality_filters(features_df)
    features_df = prune_sparse_sources(features_df, min_windows_per_source=MIN_WINDOWS_PER_SOURCE)
    features_df = filter_classwise_hrv_outliers(features_df, q_low=CLASSWISE_Q_LOW, q_high=CLASSWISE_Q_HIGH)
    features_df = deduplicate_features(features_df)
    features_df = cap_source_dominance(features_df, max_share_per_label=MAX_SOURCE_SHARE_PER_LABEL)
    features_df = rebalance_labels_datafirst(features_df, target_ratio=TARGET_LABEL_IMBALANCE_MAX)

# Mode 2: preprocessed npy splits
elif all((PREPROCESSED_DIR / f).exists() for f in ["X_train.npy", "y_train.npy", "X_test.npy", "y_test.npy"]):
    print(
        "Canh bao: thieu cot raw waveform (Time, Voltage, Peak). "
        "Dang train truc tiep tu X_*.npy va y_*.npy trong preprocessed.",
    )
    features_df = load_from_preprocessed_npy(PREPROCESSED_DIR)
    if features_df.empty:
        raise ValueError("Khong tao duoc feature dataframe tu npy splits.")

# Mode 3: fallback feature table
else:
    print(
        "Canh bao: khong co raw waveform va cung khong du npy splits. "
        "Chuyen sang su dung feature file co san.",
    )
    fallback_df = load_feature_candidates()
    if fallback_df.empty:
        raise ValueError(
            "Khong tim thay du lieu de train. Can it nhat mot trong 3 mode: "
            "(1) raw waveform co Time/Voltage/Peak, (2) X_*.npy + y_*.npy, (3) feature csv co cot label.",
        )
    features_df = fallback_df.copy()

features_df = standardize_features_df(features_df)
features_df.to_csv(FEATURE_FILE, index=False)
print(f"\nSaved features file: {FEATURE_FILE}")
print(f"Feature shape: {features_df.shape}")
print("\nClass distribution (label):")
print(features_df["label"].value_counts(dropna=False).sort_index())
if "source_file" in features_df.columns:
    print("\nUnique source files per label:")
    print(features_df.groupby("label")["source_file"].nunique().sort_index())
display(features_df.head())

Da tim thay du lieu raw waveform -> trich xuat feature HRV (data-first mode).
Using sampled sources for feature extraction: 240 files
Skip log: {'low_window_duration_or_length': 101, 'too_few_peaks': 0, 'too_few_rr_after_filter': 0}
Quality filter: removed 0/3102 windows (0.00%)
Prune sparse sources: removed 0 rows (min_windows_per_source=4)
Classwise HRV outlier filter: removed 342 rows (q=[0.02, 0.98])
Deduplicate features: removed 371/2760 rows (13.44%)
Cap source dominance: removed 0 rows
Label rebalance (data-first): removed 0 rows, new imbalance=1.038

Saved features file: C:\Users\buck\Napplee\StressClassification\data\features\features_hrv.csv
Feature shape: (2389, 24)

Class distribution (label):
label
0    591
1    608
2    604
3    586
Name: count, dtype: int64

Unique source files per label:
label
0    60
1    60
2    60
3    60
Name: source_file, dtype: int64


,source_file,window_id,label,mean_rr_ms,median_rr_ms,sdnn_ms,std_rr_ms,rmssd_ms,pnn50,heart_rate_bpm,...,voltage_mean,voltage_std,voltage_min,voltage_max,voltage_skew,voltage_kurtosis,lf_power,hf_power,lf_hf_ratio,total_power
0,concatenated/pure_3/concat_3_012.csv,11,3,603.633996,601.562500,38.088,38.087977,18.572,0.000,99.398,...,0.180453,0.222849,-0.246057,1.077314,1.332756,2.911622,2500.147013,268.820206,9.300,2769.373062
1,concatenated/pure_3/concat_3_074.csv,9,3,622.416178,623.046875,40.161,40.160647,19.220,1.571,96.399,...,0.188787,0.215843,-0.225591,1.057756,1.348819,3.000682,2832.490064,258.927013,10.939,3096.576252
2,concatenated/mixed_2/mixed_2_010.csv,0,2,715.966504,714.843750,70.404,70.404085,48.427,30.723,83.803,...,0.219114,0.205534,-0.204841,1.058959,1.279049,2.888865,6952.372911,2085.341531,3.334,9039.669974
3,concatenated/pure_2/concat_2_106.csv,6,2,690.706286,687.500000,89.189,89.189393,85.160,37.791,86.868,...,0.196065,0.217854,-0.247479,1.056555,1.194123,2.562684,6525.966986,4704.565390,1.387,11323.211282
4,concatenated/pure_2/concat_2_024.csv,0,2,691.428829,687.500000,65.560,65.559887,43.405,26.744,86.777,...,0.214987,0.207883,-0.210600,1.050990,1.261661,2.814238,6109.393259,1792.434130,3.408,7903.544739


In [5]:
# Add Stroop-style derived features (normalized LF/HF, NN50 count proxy)
if 'features_df' in globals() and not features_df.empty:
    if {'lf_power', 'hf_power'}.issubset(features_df.columns):
        denom = features_df['lf_power'].fillna(0) + features_df['hf_power'].fillna(0)
        with np.errstate(divide='ignore', invalid='ignore'):
            features_df['lf_nu'] = np.where(denom > 0, (features_df['lf_power'] / denom) * 100.0, np.nan)
            features_df['hf_nu'] = np.where(denom > 0, (features_df['hf_power'] / denom) * 100.0, np.nan)
    if {'pnn50', 'n_peaks'}.issubset(features_df.columns) and 'nn50_count' not in features_df.columns:
        # Approximate NN50 count from pNN50 (%) and number of RR intervals
        rr_count = features_df['n_peaks'].clip(lower=1) - 1
        features_df['nn50_count'] = np.round((features_df['pnn50'] / 100.0) * rr_count).astype('float')


In [6]:
# Stroop-style sanity check: label-wise HRV trends vs expected directions
if 'features_df' not in globals() or features_df.empty:
    raise ValueError("features_df is missing. Run feature extraction first.")

df_check = features_df.copy()

# Ensure derived features exist for comparison
if {'lf_power', 'hf_power'}.issubset(df_check.columns) and 'lf_nu' not in df_check.columns:
    denom = df_check['lf_power'].fillna(0) + df_check['hf_power'].fillna(0)
    with np.errstate(divide='ignore', invalid='ignore'):
        df_check['lf_nu'] = np.where(denom > 0, (df_check['lf_power'] / denom) * 100.0, np.nan)
        df_check['hf_nu'] = np.where(denom > 0, (df_check['hf_power'] / denom) * 100.0, np.nan)
if {'pnn50', 'n_peaks'}.issubset(df_check.columns) and 'nn50_count' not in df_check.columns:
    rr_count = df_check['n_peaks'].clip(lower=1) - 1
    df_check['nn50_count'] = np.round((df_check['pnn50'] / 100.0) * rr_count).astype('float')

stroop_cols = [
    'mean_rr_ms',
    'heart_rate_bpm',
    'sdnn_ms',
    'rmssd_ms',
    'nn50_count',
    'pnn50',
    'lf_nu',
    'hf_nu',
    'lf_hf_ratio',
 ]
missing_cols = [c for c in stroop_cols if c not in df_check.columns]
if missing_cols:
    raise ValueError(f"Missing Stroop-style columns: {missing_cols}")

df_check = df_check.dropna(subset=['label']).copy()
df_check['label'] = df_check['label'].astype(int)

# Label-wise summary (mean/std/median/count)
summary = df_check.groupby('label')[stroop_cols].agg(['mean', 'std', 'median', 'count']).round(3)
print("\nLabel-wise Stroop feature summary:")
display(summary)

# Trend check vs expected Stroop direction
expected_sign = {
    'mean_rr_ms': -1,
    'heart_rate_bpm': 1,
    'sdnn_ms': -1,
    'rmssd_ms': -1,
    'nn50_count': -1,
    'pnn50': -1,
    'lf_nu': 1,
    'hf_nu': -1,
    'lf_hf_ratio': 1,
}
trend_rows = []
labels_sorted = sorted(df_check['label'].unique().tolist())
for col in stroop_cols:
    corr = df_check['label'].corr(df_check[col], method='spearman')
    corr_val = float(corr) if pd.notna(corr) else np.nan
    exp = expected_sign.get(col, 0)
    ok = (np.sign(corr_val) == np.sign(exp)) if pd.notna(corr_val) else False
    trend_rows.append({
        'feature': col,
        'spearman_corr(label, feature)': round(corr_val, 3) if pd.notna(corr_val) else np.nan,
        'expected_direction': 'up' if exp > 0 else 'down',
        'matches_expected': bool(ok),
    })

trend_df = pd.DataFrame(trend_rows)
print("\nTrend check vs expected Stroop directions:")
display(trend_df)

# Mixed vs pure composition per label (to detect concat-driven shifts)
if 'source_file' in df_check.columns:
    src = df_check['source_file'].astype(str).str.lower()
    df_check['source_type'] = np.where(src.str.contains('mixed'), 'mixed', np.where(src.str.contains('pure'), 'pure', 'other'))
    comp = pd.crosstab(df_check['label'], df_check['source_type'])
    print("\nSource composition by label:")
    display(comp)


Label-wise Stroop feature summary:


mean_rr_ms                        heart_rate_bpm                       \
            mean     std   median count           mean    std  median count   
label                                                                         
0        681.728  52.918  676.502   591         88.531  6.726  88.692   591   
1        687.664  52.991  682.645   608         87.765  6.695  87.893   608   
2        662.688  46.409  657.176   604         90.982  6.326  91.300   604   
3        633.563  51.792  627.464   586         95.327  7.683  95.623   586   

      sdnn_ms          ...   lf_nu         hf_nu                        \
         mean     std  ...  median count    mean     std  median count   
label                  ...                                               
0      69.661  17.940  ...  55.803   591  42.412  12.959  44.197   591   
1      67.046  18.066  ...  60.482   608  36.557  12.410  39.518   608   
2      60.842  15.156  ...  65.087   604  31.528  11.532  34.913   604   
3      55.283  16.250  ...  70.657   586  26.260  10.263  29.343   586   

      lf_hf_ratio                      
             mean    std median count  
label                                  
0           1.668  1.066  1.263   591  
1           2.216  1.489  1.530   608  
2           2.843  1.945  1.864   604  
3           3.810  2.757  2.408   586  

[4 rows x 36 columns]


Trend check vs expected Stroop directions:


,feature,"spearman_corr(label, feature)",expected_direction,matches_expected
0,mean_rr_ms,-0.321,down,True
1,heart_rate_bpm,0.321,up,True
2,sdnn_ms,-0.302,down,True
3,rmssd_ms,-0.366,down,True
4,nn50_count,-0.512,down,True
5,pnn50,-0.512,down,True
6,lf_nu,0.456,up,True
7,hf_nu,-0.456,down,True
8,lf_hf_ratio,0.456,up,True



Source composition by label:


source_type,mixed,pure
label,,
0,116,475
1,135,473
2,65,539
3,134,452
